# Harmony360 Core Reconciliation — HarmonicCryptoVault v1

**Purpose:** bring the historical `HarmonicCryptoVault` name up to a legitimate encrypted-storage implementation while preserving Harmony360 lineage.

Reconciliation doctrine:

`Name → Meaning → Claim Type → Contract → Reference Implementation → Controls → Tests → Evidence → Scope of Conclusion`

This notebook:
- uses AES-256-GCM authenticated encryption;
- never uses φ/π as a replacement for a cryptographic key;
- treats Harmony360's contribution as artifact identity, lineage, policy, evidence, and reconstruction metadata;
- includes a governed key registry;
- includes tamper/wrong-key/round-trip tests;
- writes a reconciliation evidence record.


In [ ]:
# Colab setup
!pip -q install cryptography

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from __future__ import annotations

import os
import json
import base64
import hashlib
import secrets
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Callable

from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag

# Try to inherit the user's current Harmony360 class if available.
# If not, use a minimal shim so the notebook remains independently executable.
try:
    from harmony360_core import Harmony360  # adjust import if your canonical package path differs
    CORE_IMPORT = "harmony360_core.Harmony360"
except Exception:
    class Harmony360:
        def __init__(self):
            self.phi = (1 + 5 ** 0.5) / 2
            self.pi = 3.141592653589793
            self.alpha = 1 / 137
    CORE_IMPORT = "local_notebook_shim"

print("Harmony360 base:", CORE_IMPORT)


In [ ]:
class Harmony360KeyRegistry:
    """Development key registry.

    Production note:
    Replace in-memory key storage with an external secret/KMS provider.
    The artifact stores only key_id, never the raw AES key.
    """

    def __init__(self):
        self._keys: dict[str, bytes] = {}
        self._status: dict[str, str] = {}

    def create_key(self, key_id: str | None = None) -> str:
        key_id = key_id or f"h360-key-{secrets.token_hex(8)}"
        if key_id in self._keys:
            raise ValueError(f"Duplicate key_id: {key_id}")
        self._keys[key_id] = AESGCM.generate_key(bit_length=256)
        self._status[key_id] = "ACTIVE"
        return key_id

    def resolve(self, key_id: str) -> bytes:
        if self._status.get(key_id) != "ACTIVE":
            raise KeyError(f"Key is unavailable or inactive: {key_id}")
        return self._keys[key_id]

    def revoke(self, key_id: str) -> None:
        if key_id not in self._keys:
            raise KeyError(key_id)
        self._status[key_id] = "REVOKED"

    def status(self, key_id: str) -> str | None:
        return self._status.get(key_id)


In [ ]:
class HarmonicCryptoVault(Harmony360):
    """Harmony360 authenticated encrypted-storage component.

    Cryptographic foundation: AES-256-GCM.
    Harmony360 layer: artifact identity, lineage, lifecycle/policy metadata,
    evidence references, and reconstruction metadata.

    No φ/π transform is treated as a cryptographic primitive.
    """

    FORMAT = "HARMONY360_CRYPTO_VAULT"
    VERSION = "1.0"
    ALGORITHM = "AES-256-GCM"
    NONCE_BYTES = 12
    KEY_BYTES = 32

    def __init__(self, vault_dir: str | Path, key_resolver: Callable[[str], bytes]):
        super().__init__()
        self.vault_dir = Path(vault_dir)
        self.vault_dir.mkdir(parents=True, exist_ok=True)
        self.key_resolver = key_resolver

    @staticmethod
    def _canonical_json(value: Any) -> bytes:
        return json.dumps(
            value, sort_keys=True, separators=(",", ":"), ensure_ascii=False
        ).encode("utf-8")

    @staticmethod
    def _sha256(data: bytes) -> str:
        return hashlib.sha256(data).hexdigest()

    def validate_resonance_key(self, key_id: str) -> bool:
        try:
            key = self.key_resolver(key_id)
        except Exception:
            return False
        return isinstance(key, bytes) and len(key) == self.KEY_BYTES

    def store_encrypted_har360(
        self,
        data_blob: Any,
        key_id: str,
        artifact_id: str,
        metadata: dict | None = None,
    ) -> Path:
        if not self.validate_resonance_key(key_id):
            raise ValueError("Invalid Harmony360 vault key.")

        key = self.key_resolver(key_id)
        plaintext = self._canonical_json(data_blob)
        plaintext_hash = self._sha256(plaintext)
        nonce = os.urandom(self.NONCE_BYTES)

        header = {
            "format": self.FORMAT,
            "version": self.VERSION,
            "algorithm": self.ALGORITHM,
            "artifact_id": artifact_id,
            "key_id": key_id,
            "plaintext_sha256": plaintext_hash,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "metadata": metadata or {},
        }

        aad = self._canonical_json(header)
        ciphertext = AESGCM(key).encrypt(nonce, plaintext, aad)

        envelope = {
            "header": header,
            "nonce_b64": base64.b64encode(nonce).decode("ascii"),
            "ciphertext_b64": base64.b64encode(ciphertext).decode("ascii"),
            "ciphertext_sha256": self._sha256(ciphertext),
        }

        destination = self.vault_dir / f"{artifact_id}.encrypted.har360"
        destination.write_text(
            json.dumps(envelope, indent=2, sort_keys=True), encoding="utf-8"
        )
        return destination

    def load_encrypted_har360(self, path: str | Path, key_id: str) -> Any:
        path = Path(path)
        envelope = json.loads(path.read_text(encoding="utf-8"))
        header = envelope["header"]

        if header["format"] != self.FORMAT:
            raise ValueError("Not a Harmony360 CryptoVault artifact.")
        if header["version"] != self.VERSION:
            raise ValueError("Unsupported vault version.")
        if header["algorithm"] != self.ALGORITHM:
            raise ValueError("Unexpected encryption algorithm.")
        if header["key_id"] != key_id:
            raise ValueError("Key identifier mismatch.")
        if not self.validate_resonance_key(key_id):
            raise ValueError("Invalid Harmony360 vault key.")

        nonce = base64.b64decode(envelope["nonce_b64"])
        ciphertext = base64.b64decode(envelope["ciphertext_b64"])

        if self._sha256(ciphertext) != envelope["ciphertext_sha256"]:
            raise ValueError("Ciphertext hash mismatch.")

        try:
            plaintext = AESGCM(self.key_resolver(key_id)).decrypt(
                nonce, ciphertext, self._canonical_json(header)
            )
        except InvalidTag as exc:
            raise ValueError(
                "Authentication failed: artifact, metadata, key, nonce, "
                "or ciphertext may have been altered."
            ) from exc

        if self._sha256(plaintext) != header["plaintext_sha256"]:
            raise ValueError("Decrypted payload hash mismatch.")

        return json.loads(plaintext.decode("utf-8"))


In [ ]:
# Configure output paths
ROOT = Path("/content/drive/MyDrive/HAR360_MASTER")
RUN_ROOT = ROOT / "reconciliation" / "crypto_vault"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

registry = Harmony360KeyRegistry()
key_id = registry.create_key("h360-key-demo-v1")
vault = HarmonicCryptoVault(RUN_ROOT / "vault", registry.resolve)

sample = {
    "type": "HAR360_RECONCILIATION_TEST",
    "message": "Harmony360 CryptoVault round-trip verification",
    "phi": vault.phi,
}

artifact_path = vault.store_encrypted_har360(
    sample,
    key_id=key_id,
    artifact_id="cryptovault_demo",
    metadata={
        "lineage": "historical HarmonicCryptoVault -> reconciliation v1",
        "claim_type": "engineering",
    },
)

restored = vault.load_encrypted_har360(artifact_path, key_id)
print("Artifact:", artifact_path)
print("Round trip:", restored == sample)


In [ ]:
# Automated verification suite
import copy

results = {}

# 1. Round trip
results["round_trip"] = vault.load_encrypted_har360(artifact_path, key_id) == sample

# 2. Wrong key
wrong_id = registry.create_key("h360-key-wrong")
try:
    vault.load_encrypted_har360(artifact_path, wrong_id)
    results["wrong_key_rejected"] = False
except Exception:
    results["wrong_key_rejected"] = True

# 3. Duplicate plaintext should produce different ciphertext due to fresh nonce
artifact_path_2 = vault.store_encrypted_har360(
    sample,
    key_id=key_id,
    artifact_id="cryptovault_demo_2",
    metadata={"lineage": "nonce uniqueness test"},
)
e1 = json.loads(artifact_path.read_text())
e2 = json.loads(artifact_path_2.read_text())
results["fresh_nonce"] = e1["nonce_b64"] != e2["nonce_b64"]
results["different_ciphertext"] = e1["ciphertext_b64"] != e2["ciphertext_b64"]

# 4. Tamper ciphertext
tampered = copy.deepcopy(e1)
raw = bytearray(base64.b64decode(tampered["ciphertext_b64"]))
raw[0] ^= 1
tampered["ciphertext_b64"] = base64.b64encode(bytes(raw)).decode("ascii")
tampered_path = RUN_ROOT / "vault" / "tampered_ciphertext.encrypted.har360"
tampered_path.write_text(json.dumps(tampered, indent=2, sort_keys=True))
try:
    vault.load_encrypted_har360(tampered_path, key_id)
    results["ciphertext_tamper_rejected"] = False
except Exception:
    results["ciphertext_tamper_rejected"] = True

# 5. Tamper authenticated metadata while preserving ciphertext hash
tampered_meta = copy.deepcopy(e1)
tampered_meta["header"]["metadata"]["claim_type"] = "altered"
tampered_meta_path = RUN_ROOT / "vault" / "tampered_metadata.encrypted.har360"
tampered_meta_path.write_text(json.dumps(tampered_meta, indent=2, sort_keys=True))
try:
    vault.load_encrypted_har360(tampered_meta_path, key_id)
    results["metadata_tamper_rejected"] = False
except Exception:
    results["metadata_tamper_rejected"] = True

# 6. Revoke key
registry.revoke(key_id)
results["revoked_key_invalid"] = not vault.validate_resonance_key(key_id)

print(json.dumps(results, indent=2))
assert all(results.values()), results
print("ALL TESTS PASS")


In [ ]:
# Persist reconciliation evidence
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
evidence = {
    "run_id": run_id,
    "capability": "HarmonicCryptoVault",
    "classification": "HARMONY360 CORE RECONCILIATION — ENGINEERING",
    "historical_intent": "encrypted .har360 storage",
    "claim_type": "engineering",
    "reference_implementation": {
        "cipher": "AES-256-GCM",
        "authenticated_metadata": True,
        "random_nonce": True,
        "tamper_detection": True,
    },
    "harmony360_extensions": [
        "artifact identity",
        "lineage metadata",
        "key identity",
        "lifecycle/policy compatibility",
        "evidence and reconstruction hooks",
    ],
    "security_boundary": {
        "phi_pi_used_as_cipher": False,
        "qrc360_used_as_security_primitive": False,
    },
    "tests": results,
    "status": "PASS" if all(results.values()) else "FAIL",
    "scope_of_conclusion": (
        "This run verifies authenticated encrypted .har360 storage behavior "
        "for the tested implementation. It does not establish a novel "
        "Harmony360 cryptographic primitive."
    ),
}

evidence_path = RUN_ROOT / f"HarmonicCryptoVault_Reconciliation_{run_id}.har360.json"
evidence_path.write_text(json.dumps(evidence, indent=2, sort_keys=True), encoding="utf-8")

print("Evidence:", evidence_path)
print(json.dumps(evidence, indent=2))


## Next reconciliation target

After this vault/key-registry foundation, the next core batch is:

1. `Harmony360Trainer`
2. `Harmony360HybridTrainer.train_language_model()`
3. `Harmony360Snapshot`
4. `HAR360LedgerManager`
5. `PhiPiTuner`

The key rule remains: **when the name claims a capability, correct the implementation until the capability is real.**
